In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)

(6362620, 11)


In [2]:
ID_COLUMNS = ["nameOrig", "nameDest"]

df_model = df.drop(columns=ID_COLUMNS).copy()

df_model.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,170136.0,160296.36,0.0,0.0,0,0
1,1,PAYMENT,1864.28,21249.0,19384.72,0.0,0.0,0,0
2,1,TRANSFER,181.00,181.0,0.00,0.0,0.0,1,0
3,1,CASH_OUT,181.00,181.0,0.00,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,41554.0,29885.86,0.0,0.0,0,0


In [3]:
df_model = pd.get_dummies(
    df_model,
    columns=["type"],
    prefix="type",
    dtype=int
)

df_model.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,0,0,0,1,0


In [4]:
df_model["orig_balance_change"] = (
    df_model["oldbalanceOrg"] -
    df_model["newbalanceOrig"]
)

df_model["dest_balance_change"] = (
    df_model["newbalanceDest"] -
    df_model["oldbalanceDest"]
)

df_model["orig_amount_gap"] = (
    df_model["orig_balance_change"] -
    df_model["amount"]
)

df_model["dest_amount_gap"] = (
    df_model["dest_balance_change"] -
    df_model["amount"]
)

In [5]:
EPSILON = 1e-6

df_model["amount_to_orig_balance"] = (
    df_model["amount"] /
    (df_model["oldbalanceOrg"] + EPSILON)
)

df_model["amount_to_dest_balance"] = (
    df_model["amount"] /
    (df_model["oldbalanceDest"] + EPSILON)
)

In [6]:
df_model["orig_balance_zero_before"] = (
    df_model["oldbalanceOrg"] == 0
).astype(int)

df_model["orig_balance_zero_after"] = (
    df_model["newbalanceOrig"] == 0
).astype(int)

df_model["dest_balance_zero_before"] = (
    df_model["oldbalanceDest"] == 0
).astype(int)

df_model["dest_balance_zero_after"] = (
    df_model["newbalanceDest"] == 0
).astype(int)

In [7]:
df_model["log_amount"] = np.log1p(df_model["amount"])

In [8]:
df_model["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [9]:
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())
print("Unique steps:", df["step"].nunique())

Minimum step: 1
Maximum step: 743
Unique steps: 743


In [10]:
df["step"].value_counts().sort_index().head()

step
1    2708
2    1014
3     552
4     565
5     665
Name: count, dtype: int64

In [11]:
df["step"].value_counts().sort_index().tail()

step
739    10
740     6
741    22
742    14
743     8
Name: count, dtype: int64

In [12]:
steps = sorted(df["step"].unique())

train_end = int(len(steps) * 0.70)
val_end = int(len(steps) * 0.85)

train_steps = steps[:train_end]
val_steps = steps[train_end:val_end]
test_steps = steps[val_end:]

train_df = df[df["step"].isin(train_steps)].copy()
val_df = df[df["step"].isin(val_steps)].copy()
test_df = df[df["step"].isin(test_steps)].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (6082007, 11)
Validation: (191147, 11)
Test: (89466, 11)


In [13]:
def fraud_summary(data, name):
    fraud_count = data["isFraud"].sum()
    total = len(data)
    fraud_rate = fraud_count / total * 100

    print(f"\n{name}")
    print("-" * 40)
    print("Transactions:", total)
    print("Fraud:", fraud_count)
    print(f"Fraud rate: {fraud_rate:.4f}%")

fraud_summary(train_df, "TRAIN")
fraud_summary(val_df, "VALIDATION")
fraud_summary(test_df, "TEST")


TRAIN
----------------------------------------
Transactions: 6082007
Fraud: 5781
Fraud rate: 0.0951%

VALIDATION
----------------------------------------
Transactions: 191147
Fraud: 1180
Fraud rate: 0.6173%

TEST
----------------------------------------
Transactions: 89466
Fraud: 1252
Fraud rate: 1.3994%


In [14]:
train_df.to_csv(
    "../data/processed/train.csv",
    index=False
)

val_df.to_csv(
    "../data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False
)